## Setup

In [ ]:
import pandas as pd  # tables of data

data_path = "/content/drive/My Drive/neet_data.csv"  # where the cleaned data lives on Drive

## Mount Google Drive

In [ ]:
from google.colab import drive  # Colab's Google Drive tool
drive.mount("/content/drive")  # connect Google Drive

## Load the data

In [ ]:
df = pd.read_csv(data_path)  # load the cleaned data into a table

## 2026: Education of NEET people by age group and sex

In [ ]:
dummies = {"less_hs": "Less than high school", "hs_ged": "High school or GED", "some_college": "Some college", "bachelor_plus": "Bachelor or higher"}  # education dummies and their labels, in order
age_groups = ["16-19", "20-24", "25-29"]  # age groups, in order
row_order = [(age, sex) for age in age_groups for sex in ["Men", "Women"]]  # the six rows: Men before Women in each age group

df["agegrp"] = pd.cut(df["age"], bins=[16, 20, 25, 30], labels=age_groups, right=False)  # age groups from age: 16-19, 20-24, 25-29 (both ends included)
df["sex"] = df["female"].map({0: "Men", 1: "Women"})  # words for female: 1 = Women, 0 = Men

neet26 = df[(df["year"] == 2026) & (df["neet"] == 1)].copy()  # keep only 2026 and only NEET people
n_left_out = neet26[list(dummies)].isna().any(axis=1).sum()  # NEET people with missing education
known = neet26.dropna(subset=list(dummies))  # leave them out

percent = {}  # weighted percent, one row per age group and sex
people = {}  # unweighted respondents, same layout
for age, sex in row_order:  # loop over the six rows
    group = known[(known["agegrp"] == age) & (known["sex"] == sex)]  # NEET people in this row
    total_weight = group["asecwt"].sum()  # weighted number of NEET people in this row
    percent[f"{age} {sex}"] = {}  # start this row of the table
    people[f"{age} {sex}"] = {}  # start this row of the counts
    for var, label in dummies.items():  # loop over the four education dummies
        percent[f"{age} {sex}"][label] = (group[var] * group["asecwt"]).sum() / total_weight * 100  # weighted percent with this education
        people[f"{age} {sex}"][label] = int((group[var] == 1).sum())  # unweighted respondents with this education

table = pd.DataFrame(percent).T  # dictionary -> table, one row per age group and sex
people = pd.DataFrame(people).T  # same layout for the counts

shown = table.copy().astype(object)  # text version of the table, for the asterisks
for row in table.index:  # loop over the rows
    for column in table.columns:  # loop over the columns
        star = "*" if people.loc[row, column] < 30 else " "  # asterisk if fewer than 30 respondents
        shown.loc[row, column] = f"{table.loc[row, column]:.1f}{star}"  # percent with 1 decimal, then the asterisk or a space

row_sums = table.sum(axis=1)  # each row's total, before rounding
off = (row_sums - 100).abs()  # distance from 100 in each row

print("NEET people in 2026 left out because education is missing (unweighted):", n_left_out)  # how many were left out
print()  # empty line
print("Table. Education of NEET people in 2026, % of NEET people in each row (weighted with asecwt).")  # table title
print(shown.to_string())  # table, 1 decimal, asterisks
print("* fewer than 30 unweighted respondents")  # key for the asterisk
print()  # empty line
print("Check: every row adds up to 100.0:", bool((off < 1e-9).all()))  # True or False
print("Largest distance from 100 in any row (before rounding):", off.max())  # should be about 0
print()  # empty line
print("Unweighted number of respondents in each cell.")  # counts title
print(people.to_string())  # counts, same layout

**What one cell means.** Take the row "20-24 Women" and the column "Some college". Among women aged 20-24 in the 2026 survey who were NEET (not employed and not in school) and whose education is known, the cell is the percent who had some college. Each row adds up to 100, so the cells show how NEET people in that row are split across the four education levels. This is a share of NEET people, not a share of all women aged 20-24, so it is not a NEET rate. People are counted by how many people in the country they represent (`asecwt`). An asterisk marks a cell with fewer than 30 survey respondents behind it, shown in the count table under the table.